# Pilot — Part A (P1–P4c)
**Settings (right panel):** Accelerator **GPU T4 x2** (never P100) · Internet **On** · no secrets needed.
Run the cells top to bottom. **If a cell fails, stop and send me its full output** — do not re-run cells in a different order.
At the end, download `pilot_part_a_results.zip` from the Output panel and send it to me, together with the output of the summary cell.
Status of everything this notebook produces: **pilot only — never reported as results.**

In [ ]:
# Cell 1 — hardware check (must show Tesla T4)
!nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv
!python --version
!nproc && free -g && df -h /kaggle/working

In [ ]:
# Cell 2 — code + environment + tests (~3-5 min)
%cd /kaggle/working
!git clone https://github.com/adeelcheena791/rag-embed-transfer.git rag
%cd /kaggle/working/rag
!git log --oneline -1
!pip install -q -r requirements-retrieval.txt
!python -m src.env_report --session pilot_A --out results/raw/env_pilot_A.json
!python -m pytest -q 2>&1 | tail -3

In [ ]:
# Cell 3 — P1: download the 5 pinned datasets (~1 GB) and verify every commit sha
!python -m src.data.download --config configs/pilot.yaml

In [ ]:
# Cell 4 — P3: join BEIR-NQ queries to NQ-open answers
# Gate (addendum v0.2.2): n_joined >= 1000. The join rate is reported, not a gate.
!python -m src.data.join_nq --config configs/pilot.yaml

In [ ]:
# Cell 4b — D1a: describe the queries that did not join (descriptive only, changes nothing)
!python -m src.data.join_diagnostics --config configs/pilot.yaml

In [ ]:
# Cell 5 — P3b: BM25 over the FULL 2.68M-passage corpus for the 200 pilot queries, then build the
# 50,000-passage pilot corpus (the BM25 run is found automatically). CPU + RAM heavy: ~10-30 min (UNVERIFIED).
!python -m src.retrieval.bm25 --config configs/pilot.yaml --corpus full --queries order:200
!python -m src.data.subcorpus --config configs/pilot.yaml --size 50000 --n-queries 200 --name pilot

In [ ]:
# Cell 6 — P2: SciFact pipeline sanity check, all 6 dense models (pass = within 0.02 of the model card)
!CUDA_VISIBLE_DEVICES=0 python -m src.sanity.scifact --config configs/pilot.yaml --models R1,R2,R3,R4,R5,R6

In [ ]:
# Cell 7 — P4c: R6 (Qwen3-Embedding-0.6B) smoke test on the T4 — 100 passages
!CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/pilot.yaml --models R6 --corpus pilot --max-passages 100

In [ ]:
# Cell 7b — ONLY IF Cell 7 printed "R6 FAILED" or "R6 DROPPED": F3 CPU projection (decides run-on-CPU vs drop R6)
# Otherwise skip this cell.
# !python -m src.retrieval.throughput --config configs/pilot.yaml --models R6 --n 2000 --device cpu

In [ ]:
# Cell 8 — P4: throughput benchmark -> pre-registered full-corpus vs sub-corpus decision (~20-40 min, UNVERIFIED)
# If R6 failed in Cell 7, remove ",R6" from --models.
!CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/pilot.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128

In [ ]:
# Cell 9 — P4b: encode the pilot corpus with R1 and R4, then exact search for the 200 pilot queries
!CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/pilot.yaml --models R1,R4 --corpus pilot
!python -m src.retrieval.search --config configs/pilot.yaml --model R1 --corpus pilot --queries order:200 --latency
!python -m src.retrieval.search --config configs/pilot.yaml --model R4 --corpus pilot --queries order:200 --latency

In [ ]:
# Cell 10 — summary: COPY THIS WHOLE OUTPUT INTO THE CHAT
!python -m src.pilot_summary --config configs/pilot.yaml

In [ ]:
# Cell 11 — package raw results + logs for download (Output panel -> pilot_part_a_results.zip)
!zip -qr /kaggle/working/pilot_part_a_results.zip results logs configs/resolved_revisions.json data/processed/nq_join_stats.json data/processed/nq_join_diagnostics.json data/processed/subcorpus_pilot_ids.txt
!ls -la /kaggle/working/pilot_part_a_results.zip